# Lab 3.10 &mdash; Text Generation: Greedy vs Sampling

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Generate text with a real GPT-style model
- Compare greedy decoding vs temperature sampling
- Run the same task on a hosted 'GPT API' (ChatGroq)

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; Encoder vs decoder (BERT vs GPT)](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = "/tmp/biaa-lab-03-10"
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
GPT-style models **generate** by repeatedly predicting the **next token** and feeding it back in. Two
decoding strategies: **greedy** (always the most likely token &mdash; deterministic, can get repetitive)
and **sampling** with a **temperature** (lower = safer/sharper, higher = more random/creative). We run
the real **distilgpt2** locally, then the client's "GPT API" framing for real via **ChatGroq**
(`openai/gpt-oss-20b`). Same loop &mdash; local tiny model vs a large hosted one.

## Build it
Fill in the model and the two decoding knobs.

In [ ]:
from transformers import pipeline, set_seed
def build_gen():
    set_seed(0)
    return pipeline("text-generation", model=___)   # TODO: "distilgpt2"

def greedy(gen, prompt, n=25):
    out = gen(prompt, max_new_tokens=n, do_sample=___)   # TODO: False -- always the top token
    return out[0]["generated_text"]

def sampled(gen, prompt, n=25, temp=1.0):
    out = gen(prompt, max_new_tokens=n, do_sample=True, top_k=50, temperature=___)   # TODO: temp
    return out[0]["generated_text"]

## Run it for real
Generate locally with distilgpt2 &mdash; greedy vs low vs high temperature.

In [ ]:
try:
    gen = build_gen()
    prompt = "The future of artificial intelligence is"
    print("GREEDY      :", greedy(gen, prompt))
    print()
    print("TEMP 0.7    :", sampled(gen, prompt, temp=0.7))
    print()
    print("TEMP 1.5    :", sampled(gen, prompt, temp=1.5))
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## Run it for real
The same task on a hosted 'GPT API' via ChatGroq. (One-line swap to OpenAI shown in a comment.)

In [ ]:
try:
    import os
    if not os.environ.get("GROQ_API_KEY"):
        print("Set GROQ_API_KEY in .env to run the hosted model (this cell is optional).")
    else:
        from langchain_groq import ChatGroq
        llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.7)
        reply = llm.invoke("Continue in one vivid sentence: The future of artificial intelligence is")
        print("GROQ (gpt-oss-20b):", reply.content)
    # One-line swap to OpenAI instead of Groq:
    #   from langchain_openai import ChatOpenAI
    #   llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)   # needs OPENAI_API_KEY
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__, "--", e)

## What to notice
- **Greedy** is deterministic and can loop / repeat; **temperature 0.7** is coherent-but-varied; **1.5** is wilder and less grammatical.
- distilgpt2 is tiny, so its local text is rough &mdash; that is honest. The **hosted** model (Groq) is far more fluent: **same decoding loop, far more scale**.
- `temperature` on any chat API is exactly this softmax-sharpness knob you are now setting.

## Your turn (open task &mdash; no grader)
Sweep temperature from `0.2` to `2.0` on your own prompt. Where does it stop being coherent?
Then compare distilgpt2's local output to the Groq output on the *same* prompt. A "good" answer: you
can describe, in one sentence each, what low vs high temperature does and where scale matters most.

---
### Key takeaway
Greedy + temperature are the decoding knobs behind every chat model -- local or hosted. You now know what 'temperature' on the GPT API actually does.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>